# PEFT sweep — smoke test

Smoke the sweep **before** the full 4-cell run

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
if not os.path.exists('manage.py') and not os.path.isdir('Style-Aware-MT'):
    !git clone --branch feat/peft-implementation https://github.com/prnamhr/Style-Aware-MT.git
%cd Style-Aware-MT
!git rev-parse --short HEAD

In [ ]:
!pip install -q "transformers==5.12.1" "peft==0.18.0" "accelerate==1.14.0" \
  "bitsandbytes==0.49.2" "sentence-transformers==5.5.1" "sacrebleu==2.6.0" "PyYAML==6.0.3"

## Step 1 — dry-run the plan

Prints the candidate grid. Confirm α/r = 2 holds across ranks and the (r, lr, epoch)

In [ ]:
!python -m src.peft.sweep --dry-run

## Step 2 — chat-template masking sanity (installed tokenizer, no training)


In [ ]:
import json
from transformers import AutoTokenizer
from src.peft.train import build_example, SFTDataset

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
style = open("prompts/style_instruction.txt").read()
row = next(json.loads(l) for l in open("data/splits/train.jsonl") if l.strip())

ex = build_example(tok, style, row["input"], row["output"], max_seq_len=1024)
n_sup  = sum(1 for x in ex["labels"] if x != -100)
n_mask = sum(1 for x in ex["labels"] if x == -100)
print(f"prompt-masked tokens: {n_mask}   supervised target tokens: {n_sup}   total: {len(ex['labels'])}")
assert n_sup > 0 and n_mask > 0, "masking failed: expected both prompt-masked and supervised tokens"
print("supervised tail decodes to:", repr(tok.decode([x for x in ex['labels'] if x != -100])[:120]))

# drop-guard: an impossibly small window fully masks the example -> dropped + warned
ds = SFTDataset([row], tok, style, max_seq_len=8)
assert len(ds) == 0, "fully-masked example should have been dropped"
print("\nMASKING OK")

## Step 3 — build a tiny smoke config from `peft_sweep.yaml`

In [ ]:
import yaml
USE_4BIT = False   # set True on a <=16GB GPU (Colab T4); trains + generates in 4bit

cfg = yaml.safe_load(open("configs/peft_sweep.yaml"))
cfg["data"]["limit"] = 32                        # 32 train / 32 val
cfg["peft"]["train"]["num_train_epochs"] = 1     # one epoch -> ~5 min
cfg["peft"]["load_in_4bit"] = USE_4BIT
cfg["generator"]["load_in_4bit"] = USE_4BIT      # keep generation on the same footprint
cfg["sweep"]["grid"] = [{"r": 16, "alpha": 32, "lr": 2.0e-4, "anchor": True}]  # anchor only

with open("configs/peft_smoke.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print(open("configs/peft_smoke.yaml").read())

## Step 4 — target centroid, then one cell end-to-end

In [ ]:
!python -m src.eval.stylometrics --build-centroid

In [ ]:
!python -m src.peft.sweep --config configs/peft_smoke.yaml --epochs-keep -1

## Step 5 — verify the plumbing artifacts

In [ ]:
import json, glob
from pathlib import Path
import yaml
from src.peft.sweep import enumerate_candidates

cfg = yaml.safe_load(open("configs/peft_smoke.yaml"))
cell_dir = Path("models/peft_lora_r16_lr2e-4")

# 1) adapter saved + all-linear resolved to concrete linear module names
ac = json.loads((cell_dir / "adapter_config.json").read_text())
tm = ac["target_modules"]
print("adapter target_modules:", sorted(tm) if isinstance(tm, list) else tm)
assert isinstance(tm, list) and any("proj" in m for m in tm), "all-linear did not resolve"

# 2) epoch manifest written + checkpoints enumerable
man = json.loads((cell_dir / "epoch_checkpoints.json").read_text())
print("epoch manifest:", man)
cands = enumerate_candidates(cfg, cfg["sweep"]["output_base"])
print("enumerated candidates:", [c["tag"] for c in cands])
assert man and cands and all(Path(c["checkpoint"]).exists() for c in cands)

# 3) val generations produced (one file per kept epoch)
pred_files = sorted(glob.glob("outputs/peft_sweep/peft_r16_lr2e-4_e*_val.jsonl"))
print("prediction files:", pred_files)
preds = [json.loads(l) for l in open(pred_files[0])]
print(f"{len(preds)} preds; sample ->", repr(preds[0]["prediction"][:120]))
assert len(preds) == 32

# 4) scored + ranked leaderboard with a recommendation (anchor is selectable)
res = json.loads(open("results/peft_sweep_val.json").read())
c0 = res["cells"][0]
print("cell keys:", sorted(c0))
print(f"recommended={res['recommended']['tag'] if res['recommended'] else None}  "
      f"chrF={c0.get('chrF')}  register_fit={c0.get('register_fit')}  eval_loss={c0.get('eval_loss')}")
assert res["recommended"] and "chrF" in c0
print("\nSMOKE PASS: train -> save -> manifest -> enumerate -> generate -> score all survived.")